### Импорты

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import word_tokenize, pos_tag
from nltk.stem import WordNetLemmatizer, PorterStemmer
import nltk
import pandas as pd

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Masha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Masha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Masha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Masha\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Masha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [2]:
from sklearnex import patch_sklearn, config_context
patch_sklearn()

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


### Загрузка датасета

In [27]:
from datasets import load_dataset

dataset = load_dataset("emotion")
train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
train_labels = dataset["train"]["label"]
test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()


'[WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение' thrown while requesting HEAD https://huggingface.co/datasets/emotion/resolve/cab853a1dbdf4c42c2b3ef2173804746df8825fe/emotion.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since emotion couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'split' at C:\Users\Masha\.cache\huggingface\datasets\emotion\split\0.0.0\cab853a1dbdf4c42c2b3ef2173804746df8825fe (last modified on Wed Feb 18 11:25:49 2026).


### Функции предобработки

In [28]:
# raw
def preprocess_raw(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    return " ".join(tokens)

# lemmatization
def preprocess_lemma(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    lemmas = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(lemmas)

# stemming
def preprocess_stem(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    stems = [stemmer.stem(t) for t in tokens]
    return " ".join(stems)

# lemmatization + nouns/adjectives
def preprocess_lemma_nj(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tagged = pos_tag(tokens)
    filtered = [word for word, tag in tagged if tag.startswith('N') or tag.startswith('J')]
    lemmas = [lemmatizer.lemmatize(t) for t in filtered]
    return " ".join(lemmas)


### Подготовка текстов

In [29]:
train_raw = [preprocess_raw(t) for t in train_texts]
test_raw = [preprocess_raw(t) for t in test_texts]

train_lemma = [preprocess_lemma(t) for t in train_texts]
test_lemma = [preprocess_lemma(t) for t in test_texts]

train_stem = [preprocess_stem(t) for t in train_texts]
test_stem = [preprocess_stem(t) for t in test_texts]

train_lemma_nj = [preprocess_lemma_nj(t) for t in train_texts]
test_lemma_nj = [preprocess_lemma_nj(t) for t in test_texts]


### Векторизация

In [30]:
experiments = {}

# raw
vec_raw_bin = CountVectorizer(binary=True)
X_train_raw_bin = vec_raw_bin.fit_transform(train_raw)
X_test_raw_bin = vec_raw_bin.transform(test_raw)
experiments["raw_binary"] = (X_train_raw_bin, X_test_raw_bin)

vec_raw_count = CountVectorizer()
X_train_raw_count = vec_raw_count.fit_transform(train_raw)
X_test_raw_count = vec_raw_count.transform(test_raw)
experiments["raw_count"] = (X_train_raw_count, X_test_raw_count)

vec_raw_tfidf = TfidfVectorizer()
X_train_raw_tfidf = vec_raw_tfidf.fit_transform(train_raw)
X_test_raw_tfidf = vec_raw_tfidf.transform(test_raw)
experiments["raw_tfidf"] = (X_train_raw_tfidf, X_test_raw_tfidf)

# lemma
vec_lemma_bin = CountVectorizer(binary=True)
X_train_lemma_bin = vec_lemma_bin.fit_transform(train_lemma)
X_test_lemma_bin = vec_lemma_bin.transform(test_lemma)
experiments["lemma_binary"] = (X_train_lemma_bin, X_test_lemma_bin)

vec_lemma_count = CountVectorizer()
X_train_lemma_count = vec_lemma_count.fit_transform(train_lemma)
X_test_lemma_count = vec_lemma_count.transform(test_lemma)
experiments["lemma_count"] = (X_train_lemma_count, X_test_lemma_count)

vec_lemma_tfidf = TfidfVectorizer()
X_train_lemma_tfidf = vec_lemma_tfidf.fit_transform(train_lemma)
X_test_lemma_tfidf = vec_lemma_tfidf.transform(test_lemma)
experiments["lemma_tfidf"] = (X_train_lemma_tfidf, X_test_lemma_tfidf)

# stem
vec_stem_bin = CountVectorizer(binary=True)
X_train_stem_bin = vec_stem_bin.fit_transform(train_stem)
X_test_stem_bin = vec_stem_bin.transform(test_stem)
experiments["stem_binary"] = (X_train_stem_bin, X_test_stem_bin)

vec_stem_count = CountVectorizer()
X_train_stem_count = vec_stem_count.fit_transform(train_stem)
X_test_stem_count = vec_stem_count.transform(test_stem)
experiments["stem_count"] = (X_train_stem_count, X_test_stem_count)

vec_stem_tfidf = TfidfVectorizer()
X_train_stem_tfidf = vec_stem_tfidf.fit_transform(train_stem)
X_test_stem_tfidf = vec_stem_tfidf.transform(test_stem)
experiments["stem_tfidf"] = (X_train_stem_tfidf, X_test_stem_tfidf)

# lemma + NJ
vec_nj_bin = CountVectorizer(binary=True)
X_train_nj_bin = vec_nj_bin.fit_transform(train_lemma_nj)
X_test_nj_bin = vec_nj_bin.transform(test_lemma_nj)
experiments["lemma_NJ_binary"] = (X_train_nj_bin, X_test_nj_bin)

vec_nj_count = CountVectorizer()
X_train_nj_count = vec_nj_count.fit_transform(train_lemma_nj)
X_test_nj_count = vec_nj_count.transform(test_lemma_nj)
experiments["lemma_NJ_count"] = (X_train_nj_count, X_test_nj_count)

vec_nj_tfidf = TfidfVectorizer()
X_train_nj_tfidf = vec_nj_tfidf.fit_transform(train_lemma_nj)
X_test_nj_tfidf = vec_nj_tfidf.transform(test_lemma_nj)
experiments["lemma_NJ_tfidf"] = (X_train_nj_tfidf, X_test_nj_tfidf)


### Модели и функция оценки

In [31]:
models = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

def evaluate(model, X_train, X_test):
    model.fit(X_train, train_labels)
    preds = model.predict(X_test)
    f1_micro = f1_score(test_labels, preds, average="micro")
    f1_macro = f1_score(test_labels, preds, average="macro")
    f1_weighted = f1_score(test_labels, preds, average="weighted")
    return f1_micro, f1_macro, f1_weighted


### Запуск всех экспериментов

In [11]:
results = []

for exp_name, (Xtr, Xte) in experiments.items():
    print(f"\nЭксперимент: {exp_name}")

    for model_name, model in models.items():
        micro, macro, weighted = evaluate(model, Xtr, Xte)

        results.append({
            "experiment": exp_name,
            "model": model_name,
            "f1_micro": micro,
            "f1_macro": macro,
            "f1_weighted": weighted
        })

        print(
            f"{model_name:17s}"
            f" micro = {micro:.4f} "
            f" macro = {macro:.4f} "
            f" weighted = {weighted:.4f}"
        )

df_results = pd.DataFrame(results)
df_results



Эксперимент: raw_binary
DecisionTree      micro = 0.8500  macro = 0.7999  weighted = 0.8515
RandomForest      micro = 0.8740  macro = 0.8147  weighted = 0.8722
GradientBoosting  micro = 0.8440  macro = 0.8089  weighted = 0.8452
AdaBoost          micro = 0.3480  macro = 0.0900  weighted = 0.1867

Эксперимент: raw_count
DecisionTree      micro = 0.8605  macro = 0.8111  weighted = 0.8621
RandomForest      micro = 0.8760  macro = 0.8148  weighted = 0.8745
GradientBoosting  micro = 0.8405  macro = 0.7969  weighted = 0.8410
AdaBoost          micro = 0.3480  macro = 0.0900  weighted = 0.1867

Эксперимент: raw_tfidf
DecisionTree      micro = 0.8500  macro = 0.7993  weighted = 0.8509
RandomForest      micro = 0.8680  macro = 0.8024  weighted = 0.8657
GradientBoosting  micro = 0.8360  macro = 0.7994  weighted = 0.8371
AdaBoost          micro = 0.3480  macro = 0.0896  weighted = 0.1858

Эксперимент: lemma_binary
DecisionTree      micro = 0.8500  macro = 0.7998  weighted = 0.8511
RandomForest    

,experiment,model,f1_micro,f1_macro,f1_weighted
0,raw_binary,DecisionTree,0.8500,0.799911,0.851497
1,raw_binary,RandomForest,0.8740,0.814697,0.872235
2,raw_binary,GradientBoosting,0.8440,0.808903,0.845165
3,raw_binary,AdaBoost,0.3480,0.089984,0.186726
4,raw_count,DecisionTree,0.8605,0.811095,0.862072
5,raw_count,RandomForest,0.8760,0.814828,0.874465
6,raw_count,GradientBoosting,0.8405,0.796921,0.841008
7,raw_count,AdaBoost,0.3480,0.089984,0.186726
8,raw_tfidf,DecisionTree,0.8500,0.799263,0.850910
9,raw_tfidf,RandomForest,0.8680,0.802352,0.865684


### Cортировка и выбор лучшей комбинации

In [12]:
df_sorted_micro = df_results.sort_values(
    ["f1_micro", "f1_macro", "f1_weighted"],
    ascending=False
)

df_sorted_macro = df_results.sort_values(
    ["f1_macro", "f1_micro", "f1_weighted"],
    ascending=False
)

print("Топ-10 по F1 micro")
display(df_sorted_micro.head(10))

print("Топ-10 по F1 macro")
display(df_sorted_macro.head(10))

best_row = df_sorted_micro.iloc[0]
print("Лучшая комбинация по F1 micro:")
best_row


Топ-10 по F1 micro


,experiment,model,f1_micro,f1_macro,f1_weighted
5,raw_count,RandomForest,0.8760,0.814828,0.874465
13,lemma_binary,RandomForest,0.8755,0.813604,0.873910
1,raw_binary,RandomForest,0.8740,0.814697,0.872235
9,raw_tfidf,RandomForest,0.8680,0.802352,0.865684
17,lemma_count,RandomForest,0.8660,0.806399,0.864473
21,lemma_tfidf,RandomForest,0.8650,0.807255,0.863022
4,raw_count,DecisionTree,0.8605,0.811095,0.862072
16,lemma_count,DecisionTree,0.8505,0.800116,0.852180
0,raw_binary,DecisionTree,0.8500,0.799911,0.851497
12,lemma_binary,DecisionTree,0.8500,0.799848,0.851136


Топ-10 по F1 macro


,experiment,model,f1_micro,f1_macro,f1_weighted
5,raw_count,RandomForest,0.8760,0.814828,0.874465
1,raw_binary,RandomForest,0.8740,0.814697,0.872235
13,lemma_binary,RandomForest,0.8755,0.813604,0.873910
4,raw_count,DecisionTree,0.8605,0.811095,0.862072
2,raw_binary,GradientBoosting,0.8440,0.808903,0.845165
21,lemma_tfidf,RandomForest,0.8650,0.807255,0.863022
14,lemma_binary,GradientBoosting,0.8425,0.807159,0.843801
17,lemma_count,RandomForest,0.8660,0.806399,0.864473
18,lemma_count,GradientBoosting,0.8420,0.803536,0.843045
9,raw_tfidf,RandomForest,0.8680,0.802352,0.865684


Лучшая комбинация по F1 micro:


,5
experiment,raw_count
model,RandomForest
f1_micro,0.876
f1_macro,0.814828
f1_weighted,0.874465


### Подбор параметров

In [ ]:
rf_results = []

rf_params = [
    {"n_estimators": 100, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": None, "max_features": "sqrt"},
    {"n_estimators": 500, "max_depth": None, "max_features": "sqrt"},

    {"n_estimators": 200, "max_depth": 20, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 50, "max_features": "sqrt"},
    {"n_estimators": 200, "max_depth": 100, "max_features": "sqrt"},

    {"n_estimators": 200, "max_depth": None, "max_features": "log2"},
    {"n_estimators": 500, "max_depth": None, "max_features": "log2"},

    {"n_estimators": 200, "max_depth": None, "max_features": None},

    {"n_estimators": 200, "max_depth": None, "max_features": "sqrt", "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": None, "max_features": "sqrt", "min_samples_split": 5, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": None, "max_features": "sqrt", "min_samples_split": 10, "min_samples_leaf": 2},
]

for params in rf_params:
    rf = RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        **params
    )

    rf.fit(X_train_raw_count, train_labels)
    pred = rf.predict(X_test_raw_count)

    rf_results.append({
        "params": str(params),
        "f1_micro": f1_score(test_labels, pred, average="micro"),
        "f1_macro": f1_score(test_labels, pred, average="macro"),
        "f1_weighted": f1_score(test_labels, pred, average="weighted")
    })

df_rf_results = pd.DataFrame(rf_results)

print("Результаты подбора параметров RandomForest:")
print(df_rf_results.sort_values(["f1_micro", "f1_macro","f1_weighted"], ascending=False))

Результаты подбора параметров RandomForest:
                                               params  f1_micro  f1_macro  \
11  {'n_estimators': 200, 'max_depth': None, 'max_...    0.8780  0.813497   
0   {'n_estimators': 100, 'max_depth': None, 'max_...    0.8760  0.814828   
10  {'n_estimators': 200, 'max_depth': None, 'max_...    0.8755  0.811780   
8   {'n_estimators': 200, 'max_depth': None, 'max_...    0.8725  0.817856   
1   {'n_estimators': 200, 'max_depth': None, 'max_...    0.8725  0.806205   
9   {'n_estimators': 200, 'max_depth': None, 'max_...    0.8725  0.806205   
2   {'n_estimators': 500, 'max_depth': None, 'max_...    0.8690  0.804300   
5   {'n_estimators': 200, 'max_depth': 100, 'max_f...    0.8465  0.776423   
7   {'n_estimators': 500, 'max_depth': None, 'max_...    0.7855  0.653880   
6   {'n_estimators': 200, 'max_depth': None, 'max_...    0.7715  0.635420   
4   {'n_estimators': 200, 'max_depth': 50, 'max_fe...    0.7320  0.621742   
3   {'n_estimators': 200, 'max_d

### Лучшая модель

In [8]:
X_train_raw_count_dense = X_train_raw_count.toarray().astype("float32")
X_test_raw_count_dense  = X_test_raw_count.toarray().astype("float32")
best_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    max_features="sqrt",
    min_samples_split=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
with config_context(target_offload="gpu"):
    best_model.fit(X_train_raw_count_dense, train_labels)
    pred = best_model.predict(X_test_raw_count_dense)
print("F1 micro:", f1_score(test_labels, pred, average="micro"))
print("F1 macro:", f1_score(test_labels, pred, average="macro"))
print("F1 weighted:", f1_score(test_labels, pred, average="weighted"))

F1 micro: 0.878
F1 macro: 0.8134966770390286
F1 weighted: 0.87488930665247


### Настройка гиперпараметров

#### Ручная настройка 

In [21]:
params_rf = [
    {"n_estimators": 200},
    {"max_depth": None},
    {"max_features": "sqrt"},
    {"min_samples_split": 10}
]

rf_results = []
for p in params_rf:
    model = RandomForestClassifier(random_state=42, n_jobs=-1)

    model.fit(X_train_raw_count, train_labels)
    pred = model.predict(X_test_raw_count)

    rf_results.append({
        "params": str(p),
        "f1_micro": f1_score(test_labels, pred, average="micro")
    })

pd.DataFrame(rf_results).sort_values("f1_micro", ascending=False)

,params,f1_micro
0,{'n_estimators': 200},0.876
1,{'max_depth': None},0.876
2,{'max_features': 'sqrt'},0.876
3,{'min_samples_split': 10},0.876


In [15]:
dt_results = []

params_dt = [
    {"max_depth": None},
    {"max_depth": 20},
    {"max_depth": 50},
    {"max_depth": None, "min_samples_split": 10}
]

for p in params_dt:
    model = DecisionTreeClassifier(random_state=42, **p)

    model.fit(X_train_raw_count, train_labels)
    pred = model.predict(X_test_raw_count)

    dt_results.append({
        "params": str(p),
        "f1_micro": f1_score(test_labels, pred, average="micro")
    })

pd.DataFrame(dt_results).sort_values("f1_micro", ascending=False)

,params,f1_micro
0,{'max_depth': None},0.8605
3,"{'max_depth': None, 'min_samples_split': 10}",0.8580
2,{'max_depth': 50},0.4885
1,{'max_depth': 20},0.4035


In [16]:
gb_results = []

params_gb = [
    {"n_estimators": 100},
    {"n_estimators": 200},
    {"learning_rate": 0.1},
    {"learning_rate": 0.05},
    {"max_depth": 3},
    {"max_depth": 5}
]

for p in params_gb:
    model = GradientBoostingClassifier(random_state=42, **p)

    model.fit(X_train_raw_count, train_labels)
    pred = model.predict(X_test_raw_count)

    gb_results.append({
        "params": str(p),
        "f1_micro": f1_score(test_labels, pred, average="micro")
    })

pd.DataFrame(gb_results).sort_values("f1_micro", ascending=False)

,params,f1_micro
1,{'n_estimators': 200},0.8760
5,{'max_depth': 5},0.8630
0,{'n_estimators': 100},0.8405
2,{'learning_rate': 0.1},0.8405
4,{'max_depth': 3},0.8405
3,{'learning_rate': 0.05},0.7380


In [17]:
ab_results = []

params_ab = [
    {"n_estimators": 50},
    {"n_estimators": 100},
    {"learning_rate": 1.0},
    {"learning_rate": 0.5},
    {"learning_rate": 0.1}
]

for p in params_ab:
    model = AdaBoostClassifier(random_state=42, **p)

    model.fit(X_train_raw_count, train_labels)
    pred = model.predict(X_test_raw_count)

    ab_results.append({
        "params": str(p),
        "f1_micro": f1_score(test_labels, pred, average="micro")
    })

pd.DataFrame(ab_results).sort_values("f1_micro", ascending=False)

,params,f1_micro
1,{'n_estimators': 100},0.3675
3,{'learning_rate': 0.5},0.3640
0,{'n_estimators': 50},0.3620
2,{'learning_rate': 1.0},0.3620
4,{'learning_rate': 0.1},0.3510


In [9]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV, HalvingRandomSearchCV

#### GridSearchCV

In [ ]:
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 50, 100],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring="f1_micro",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_raw_count_dense, train_labels)

pred_grid_rf = grid_rf.best_estimator_.predict(X_test_raw_count_dense)

rf_grid_result = {
    "method": "GridSearchCV",
    "model": "RandomForest",
    "best_params": str(grid_rf.best_params_),
    "cv_best_score": grid_rf.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_grid_rf, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_grid_rf, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_grid_rf, average="weighted")
}

print(rf_grid_result)

Fitting 3 folds for each of 162 candidates, totalling 486 fits
{'method': 'GridSearchCV', 'model': 'RandomForest', 'best_params': "{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}", 'cv_best_score': 0.866500022393132, 'test_f1_micro': 0.8805, 'test_f1_macro': 0.820083649037839, 'test_f1_weighted': 0.8774617754669342}


In [10]:
param_grid_gb = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10]
}

gb = GradientBoostingClassifier(random_state=42)

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    scoring="f1_micro",
    cv=3,
    n_jobs=-1,
    verbose=1
)
with config_context(target_offload="gpu"):
    grid_gb.fit(X_train_raw_count_dense, train_labels)
    pred_grid_gb = grid_gb.best_estimator_.predict(X_test_raw_count_dense)

gb_grid_result = {
    "method": "GridSearchCV",
    "model": "GradientBoosting",
    "best_params": str(grid_gb.best_params_),
    "cv_best_score": grid_gb.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_grid_gb, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_grid_gb, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_grid_gb, average="weighted")
}

print(gb_grid_result)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


KeyboardInterrupt: 

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier

param_grid_hgb = {
    "max_iter": [100, 200],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5],
    "min_samples_leaf": [20]
}

hgb = HistGradientBoostingClassifier(random_state=42)

grid_hgb = GridSearchCV(
    estimator=hgb,
    param_grid=param_grid_hgb,
    scoring="f1_micro",
    cv=3,
    n_jobs=1,
    verbose=1
)

grid_hgb.fit(X_train_raw_count_dense, train_labels)
pred_grid_hgb = grid_hgb.best_estimator_.predict(X_test_raw_count_dense)

hgb_grid_result = {
    "method": "GridSearchCV",
    "model": "HistGradientBoosting",
    "best_params": str(grid_hgb.best_params_),
    "cv_best_score": grid_hgb.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_grid_hgb, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_grid_hgb, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_grid_hgb, average="weighted")
}

print(hgb_grid_result)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


KeyboardInterrupt: 

In [26]:
X_train_raw_tfidf_dense = X_train_raw_tfidf.toarray()
X_test_raw_tfidf_dense = X_test_raw_tfidf.toarray()

hgb_base = HistGradientBoostingClassifier(
    max_iter=50,
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=20,
    random_state=42
)

hgb_base.fit(X_train_raw_tfidf_dense, train_labels)
pred_hgb_base = hgb_base.predict(X_test_raw_tfidf_dense)

print("HGB without LSA (TF-IDF dense)")
print("F1 micro:", f1_score(test_labels, pred_hgb_base, average="micro"))
print("F1 macro:", f1_score(test_labels, pred_hgb_base, average="macro"))
print("F1 weighted:", f1_score(test_labels, pred_hgb_base, average="weighted"))

HGB without LSA (TF-IDF dense)
F1 micro: 0.806
F1 macro: 0.7666107829812469
F1 weighted: 0.806399481982957


#### RandomizedSearchCV

In [13]:
param_dist_rf = {
    "n_estimators": randint(100, 401),
    "max_depth": [None, 20, 50, 100],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": randint(2, 11),
    "min_samples_leaf": randint(1, 5)
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

random_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_iter=20,
    scoring="f1_micro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_rf.fit(X_train_raw_count_dense, train_labels)

pred_random_rf = random_rf.best_estimator_.predict(X_test_raw_count_dense)

rf_random_result = {
    "method": "RandomizedSearchCV",
    "model": "RandomForest",
    "best_params": str(random_rf.best_params_),
    "cv_best_score": random_rf.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_random_rf, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_random_rf, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_random_rf, average="weighted")
}

print(rf_random_result)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
{'method': 'RandomizedSearchCV', 'model': 'RandomForest', 'best_params': "{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 4, 'n_estimators': 314}", 'cv_best_score': 0.8658124794216961, 'test_f1_micro': 0.882, 'test_f1_macro': 0.8182857534541843, 'test_f1_weighted': 0.8785336093093042}


In [14]:
param_dist_gb = {
    "n_estimators": randint(50, 301),
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 7],
    "min_samples_split": randint(2, 11)
}

gb = GradientBoostingClassifier(random_state=42)

random_gb = RandomizedSearchCV(
    estimator=gb,
    param_distributions=param_dist_gb,
    n_iter=20,
    scoring="f1_micro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_gb.fit(X_train_raw_count_dense, train_labels)

pred_random_gb = random_gb.best_estimator_.predict(X_test_raw_count_dense)

gb_random_result = {
    "method": "RandomizedSearchCV",
    "model": "GradientBoosting",
    "best_params": str(random_gb.best_params_),
    "cv_best_score": random_gb.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_random_gb, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_random_gb, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_random_gb, average="weighted")
}

print(gb_random_result)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

#### HalvingGridSearchCV

In [ ]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

halving_grid_rf = HalvingGridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring="f1_micro",
    cv=3,
    factor=2,
    n_jobs=-1,
    verbose=1
)
with config_context(target_offload='gpu'):
    halving_grid_rf.fit(X_train_raw_count_dense, train_labels)
    pred_halving_grid_rf = halving_grid_rf.best_estimator_.predict(X_test_raw_count_dense)

rf_halving_grid_result = {
    "method": "HalvingGridSearchCV",
    "model": "RandomForest",
    "best_params": str(halving_grid_rf.best_params_),
    "cv_best_score": halving_grid_rf.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_halving_grid_rf, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_halving_grid_rf, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_halving_grid_rf, average="weighted")
}

print(rf_halving_grid_result)

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

halving_grid_gb = HalvingGridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    scoring="f1_micro",
    cv=3,
    factor=2,
    n_jobs=-1,
    verbose=1
)
with config_context(target_offload='gpu'):
    halving_grid_gb.fit(X_train_raw_count_dense, train_labels)
    pred_halving_grid_gb = halving_grid_gb.best_estimator_.predict(X_test_raw_count_dense)

gb_halving_grid_result = {
    "method": "HalvingGridSearchCV",
    "model": "GradientBoosting",
    "best_params": str(halving_grid_gb.best_params_),
    "cv_best_score": halving_grid_gb.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_halving_grid_gb, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_halving_grid_gb, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_halving_grid_gb, average="weighted")
}

print(gb_halving_grid_result)

#### HalvingRandomSearchCV

In [ ]:
param_dist_gb = {
    "n_estimators": randint(50, 301),
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 7],
    "min_samples_split": randint(2, 11)
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

halving_random_rf = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_dist_rf,
    n_candidates="exhaust",
    scoring="f1_micro",
    cv=3,
    factor=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
with config_context(target_offload='gpu'):
    halving_random_rf.fit(X_train_raw_count_dense, train_labels)
    pred_halving_random_rf = halving_random_rf.best_estimator_.predict(X_test_raw_count_dense)

rf_halving_random_result = {
    "method": "HalvingRandomSearchCV",
    "model": "RandomForest",
    "best_params": str(halving_random_rf.best_params_),
    "cv_best_score": halving_random_rf.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_halving_random_rf, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_halving_random_rf, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_halving_random_rf, average="weighted")
}

print(rf_halving_random_result)

NameError: name 'param_dist_rf' is not defined

In [ ]:
gb = GradientBoostingClassifier(random_state=42)

halving_random_gb = HalvingRandomSearchCV(
    estimator=gb,
    param_distributions=param_dist_gb,
    n_candidates="exhaust",
    scoring="f1_micro",
    cv=3,
    factor=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
with config_context(target_offload='gpu'):
    halving_random_gb.fit(X_train_raw_count_dense, train_labels)
    pred_halving_random_gb = halving_random_gb.best_estimator_.predict(X_test_raw_count_dense)

gb_halving_random_result = {
    "method": "HalvingRandomSearchCV",
    "model": "GradientBoosting",
    "best_params": str(halving_random_gb.best_params_),
    "cv_best_score": halving_random_gb.best_score_,
    "test_f1_micro": f1_score(test_labels, pred_halving_random_gb, average="micro"),
    "test_f1_macro": f1_score(test_labels, pred_halving_random_gb, average="macro"),
    "test_f1_weighted": f1_score(test_labels, pred_halving_random_gb, average="weighted")
}

print(gb_halving_random_result)

In [ ]:
search_compare = pd.DataFrame([
    rf_grid_result,
    rf_random_result,
    rf_halving_grid_result,
    rf_halving_random_result,
    gb_grid_result,
    gb_random_result,
    gb_halving_grid_result,
    gb_halving_random_result
])

search_compare = search_compare.sort_values(
    ["test_f1_micro", "test_f1_macro"],
    ascending=False
)

print(search_compare)

### LSA

In [15]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
import pandas as pd

#### Random Forest

In [20]:
lsa_components = [50, 100, 200, 300]

lsa_results = []

for n_comp in lsa_components:
    print(f"\nLSA components: {n_comp}")

    pipeline = Pipeline([
        ("svd", TruncatedSVD(n_components=n_comp, random_state=42)),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=2,
            min_samples_split=4,
            random_state=42,
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train_raw_tfidf, train_labels)
    pred = pipeline.predict(X_test_raw_tfidf)

    f1_micro = f1_score(test_labels, pred, average="micro")
    f1_macro = f1_score(test_labels, pred, average="macro")
    f1_weighted = f1_score(test_labels, pred, average="weighted")

    print(f"F1 micro: {f1_micro:.4f}")
    print(f"F1 macro: {f1_macro:.4f}")
    print(f"F1 weighted: {f1_weighted:.4f}")

    lsa_results.append({
        "components": n_comp,
        "model": "RandomForest + LSA",
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

lsa_df = pd.DataFrame(lsa_results).sort_values("f1_micro", ascending=False)
lsa_df


LSA components: 50
F1 micro: 0.4090
F1 macro: 0.1704
F1 weighted: 0.3208

LSA components: 100
F1 micro: 0.4105
F1 macro: 0.1746
F1 weighted: 0.3233

LSA components: 200
F1 micro: 0.4930
F1 macro: 0.2741
F1 weighted: 0.4239

LSA components: 300
F1 micro: 0.5690
F1 macro: 0.3481
F1 weighted: 0.5061


,components,model,f1_micro,f1_macro,f1_weighted
3,300,RandomForest + LSA,0.5690,0.348082,0.506101
2,200,RandomForest + LSA,0.4930,0.274094,0.423939
1,100,RandomForest + LSA,0.4105,0.174611,0.323340
0,50,RandomForest + LSA,0.4090,0.170442,0.320798


#### Gradient Boosting

In [ ]:
lsa_components = [50, 100, 200, 300]

lsa_gb_results = []

for n_comp in lsa_components:
    print(f"\nLSA components: {n_comp}")

    pipeline = Pipeline([
        ("svd", TruncatedSVD(n_components=n_comp, random_state=42)),
        ("gb", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ])

    pipeline.fit(X_train_raw_tfidf, train_labels)
    pred = pipeline.predict(X_test_raw_tfidf)

    f1_micro = f1_score(test_labels, pred, average="micro")
    f1_macro = f1_score(test_labels, pred, average="macro")
    f1_weighted = f1_score(test_labels, pred, average="weighted")

    print(f"F1 micro: {f1_micro:.4f}")
    print(f"F1 macro: {f1_macro:.4f}")
    print(f"F1 weighted: {f1_weighted:.4f}")

    lsa_gb_results.append({
        "components": n_comp,
        "model": "GradientBoosting + LSA",
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

pd.DataFrame(lsa_gb_results).sort_values("f1_micro", ascending=False)


LSA components: 50


KeyboardInterrupt: 

#### Hist Gradient Boosting

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier

lsa_components = [50, 100, 200, 300]

lsa_hgb_results = []

for n_comp in lsa_components:
    print(f"\nLSA components: {n_comp}")

    pipeline = Pipeline([
        ("svd", TruncatedSVD(n_components=n_comp, random_state=42)),
        ("hgb", HistGradientBoostingClassifier(
            max_iter=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ])

    pipeline.fit(X_train_raw_tfidf, train_labels)
    pred = pipeline.predict(X_test_raw_tfidf)

    f1_micro = f1_score(test_labels, pred, average="micro")
    f1_macro = f1_score(test_labels, pred, average="macro")
    f1_weighted = f1_score(test_labels, pred, average="weighted")

    print(f"F1 micro: {f1_micro:.4f}")
    print(f"F1 macro: {f1_macro:.4f}")
    print(f"F1 weighted: {f1_weighted:.4f}")

    lsa_hgb_results.append({
        "components": n_comp,
        "model": "HistGradientBoosting + LSA",
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

lsa_hgb_df = pd.DataFrame(lsa_hgb_results).sort_values("f1_micro", ascending=False)
lsa_hgb_df


LSA components: 50
F1 micro: 0.4145
F1 macro: 0.1881
F1 weighted: 0.3342

LSA components: 100
F1 micro: 0.4220
F1 macro: 0.2201
F1 weighted: 0.3529

LSA components: 200
F1 micro: 0.5140
F1 macro: 0.3510
F1 weighted: 0.4700

LSA components: 300
F1 micro: 0.6055
F1 macro: 0.4772
F1 weighted: 0.5765


,components,model,f1_micro,f1_macro,f1_weighted
3,300,HistGradientBoosting + LSA,0.6055,0.477208,0.576495
2,200,HistGradientBoosting + LSA,0.5140,0.351044,0.470039
1,100,HistGradientBoosting + LSA,0.4220,0.220101,0.352931
0,50,HistGradientBoosting + LSA,0.4145,0.188107,0.334241


### Части речи(POS)

In [33]:
from nltk import word_tokenize, pos_tag
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import pandas as pd

lemmatizer = WordNetLemmatizer()

def preprocess_lemma_nj(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tagged = pos_tag(tokens)
    filtered = [word for word, tag in tagged if tag.startswith('N') or tag.startswith('J')]
    lemmas = [lemmatizer.lemmatize(t) for t in filtered]
    return " ".join(lemmas)

def preprocess_lemma_njv(text):
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha()]
    tagged = pos_tag(tokens)
    filtered = [word for word, tag in tagged if tag.startswith('N') or tag.startswith('J') or tag.startswith('V')]
    lemmas = [lemmatizer.lemmatize(t) for t in filtered]
    return " ".join(lemmas)

#### Emotion

In [34]:
train_emotion_nj = [preprocess_lemma_nj(t) for t in train_texts]
test_emotion_nj = [preprocess_lemma_nj(t) for t in test_texts]

train_emotion_njv = [preprocess_lemma_njv(t) for t in train_texts]
test_emotion_njv = [preprocess_lemma_njv(t) for t in test_texts]

In [35]:
vec_emotion_nj_tfidf = TfidfVectorizer()
X_train_emotion_nj_tfidf = vec_emotion_nj_tfidf.fit_transform(train_emotion_nj)
X_test_emotion_nj_tfidf = vec_emotion_nj_tfidf.transform(test_emotion_nj)

vec_emotion_njv_tfidf = TfidfVectorizer()
X_train_emotion_njv_tfidf = vec_emotion_njv_tfidf.fit_transform(train_emotion_njv)
X_test_emotion_njv_tfidf = vec_emotion_njv_tfidf.transform(test_emotion_njv)

In [36]:
rf_pos = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features="sqrt",
    min_samples_leaf=2,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)

emotion_pos_results = []

emotion_experiments = {
    "baseline_raw_tfidf": (X_train_raw_tfidf, X_test_raw_tfidf),
    "N+J_tfidf": (X_train_emotion_nj_tfidf, X_test_emotion_nj_tfidf),
    "N+J+V_tfidf": (X_train_emotion_njv_tfidf, X_test_emotion_njv_tfidf)
}

for name, (Xtr, Xte) in emotion_experiments.items():
    rf_pos.fit(Xtr, train_labels)
    pred = rf_pos.predict(Xte)

    emotion_pos_results.append({
        "dataset": "emotion",
        "experiment": name,
        "f1_micro": f1_score(test_labels, pred, average="micro"),
        "f1_macro": f1_score(test_labels, pred, average="macro"),
        "f1_weighted": f1_score(test_labels, pred, average="weighted")
    })

df_emotion_pos = pd.DataFrame(emotion_pos_results).sort_values("f1_micro", ascending=False)
df_emotion_pos

,dataset,experiment,f1_micro,f1_macro,f1_weighted
0,emotion,baseline_raw_tfidf,0.8755,0.812385,0.871947
2,emotion,N+J+V_tfidf,0.8625,0.806438,0.860260
1,emotion,N+J_tfidf,0.7235,0.658550,0.719637


In [37]:
emotion_pos_lsa_results = []

for n_comp in [100, 300]:
    for name, Xtr, Xte in [
        ("N+J_tfidf", X_train_emotion_nj_tfidf, X_test_emotion_nj_tfidf),
        ("N+J+V_tfidf", X_train_emotion_njv_tfidf, X_test_emotion_njv_tfidf)
    ]:
        pipeline = Pipeline([
            ("svd", TruncatedSVD(n_components=n_comp, random_state=42)),
            ("rf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                max_features="sqrt",
                min_samples_leaf=2,
                min_samples_split=4,
                random_state=42,
                n_jobs=-1
            ))
        ])

        pipeline.fit(Xtr, train_labels)
        pred = pipeline.predict(Xte)

        emotion_pos_lsa_results.append({
            "dataset": "emotion",
            "experiment": name,
            "components": n_comp,
            "f1_micro": f1_score(test_labels, pred, average="micro"),
            "f1_macro": f1_score(test_labels, pred, average="macro"),
            "f1_weighted": f1_score(test_labels, pred, average="weighted")
        })

df_emotion_pos_lsa = pd.DataFrame(emotion_pos_lsa_results).sort_values("f1_micro", ascending=False)
df_emotion_pos_lsa

,dataset,experiment,components,f1_micro,f1_macro,f1_weighted
3,emotion,N+J+V_tfidf,300,0.6400,0.476643,0.602480
2,emotion,N+J_tfidf,300,0.6170,0.489145,0.592641
0,emotion,N+J_tfidf,100,0.5415,0.384366,0.506147
1,emotion,N+J+V_tfidf,100,0.5180,0.334411,0.465921


#### 20newsgroup

In [38]:
from datasets import load_dataset

dataset_news = load_dataset("SetFit/20_newsgroups")

categories = [
    'comp.sys.ibm.pc.hardware',
    'comp.sys.mac.hardware',
    'comp.graphics',
    'comp.windows.x'
]

train_news = dataset_news["train"].filter(lambda example: example["label_text"] in categories)
test_news = dataset_news["test"].filter(lambda example: example["label_text"] in categories)

train_news_texts = list(train_news["text"])
train_news_labels = list(train_news["label"])

test_news_texts = list(test_news["text"])
test_news_labels = list(test_news["label"])

'[WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение' thrown while requesting HEAD https://huggingface.co/datasets/SetFit/20_newsgroups/resolve/f1b91292074e7cfb69be58b642d583ec262f30ed/20_newsgroups.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since SetFit/20_newsgroups couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Masha\.cache\huggingface\datasets\SetFit___20_newsgroups\default\0.0.0\f1b91292074e7cfb69be58b642d583ec262f30ed (last modified on Wed Feb 18 11:59:24 2026).


In [39]:
train_news_nj = [preprocess_lemma_nj(t) for t in train_news_texts]
test_news_nj = [preprocess_lemma_nj(t) for t in test_news_texts]

train_news_njv = [preprocess_lemma_njv(t) for t in train_news_texts]
test_news_njv = [preprocess_lemma_njv(t) for t in test_news_texts]

In [40]:
vec_news_nj_tfidf = TfidfVectorizer()
X_train_news_nj_tfidf = vec_news_nj_tfidf.fit_transform(train_news_nj)
X_test_news_nj_tfidf = vec_news_nj_tfidf.transform(test_news_nj)

vec_news_njv_tfidf = TfidfVectorizer()
X_train_news_njv_tfidf = vec_news_njv_tfidf.fit_transform(train_news_njv)
X_test_news_njv_tfidf = vec_news_njv_tfidf.transform(test_news_njv)

In [41]:
news_pos_results = []

news_experiments = {
    "N+J_tfidf": (X_train_news_nj_tfidf, X_test_news_nj_tfidf),
    "N+J+V_tfidf": (X_train_news_njv_tfidf, X_test_news_njv_tfidf)
}

for name, (Xtr, Xte) in news_experiments.items():
    rf_pos.fit(Xtr, train_news_labels)
    pred = rf_pos.predict(Xte)

    news_pos_results.append({
        "dataset": "20_newsgroups",
        "experiment": name,
        "f1_micro": f1_score(test_news_labels, pred, average="micro"),
        "f1_macro": f1_score(test_news_labels, pred, average="macro"),
        "f1_weighted": f1_score(test_news_labels, pred, average="weighted")
    })

df_news_pos = pd.DataFrame(news_pos_results).sort_values("f1_micro", ascending=False)
df_news_pos

,dataset,experiment,f1_micro,f1_macro,f1_weighted
0,20_newsgroups,N+J_tfidf,0.728379,0.730392,0.730389
1,20_newsgroups,N+J+V_tfidf,0.723254,0.724772,0.724795


In [42]:
news_pos_lsa_results = []

for n_comp in [100, 300]:
    for name, Xtr, Xte in [
        ("N+J_tfidf", X_train_news_nj_tfidf, X_test_news_nj_tfidf),
        ("N+J+V_tfidf", X_train_news_njv_tfidf, X_test_news_njv_tfidf)
    ]:
        pipeline = Pipeline([
            ("svd", TruncatedSVD(n_components=n_comp, random_state=42)),
            ("rf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                max_features="sqrt",
                min_samples_leaf=2,
                min_samples_split=4,
                random_state=42,
                n_jobs=-1
            ))
        ])

        pipeline.fit(Xtr, train_news_labels)
        pred = pipeline.predict(Xte)

        news_pos_lsa_results.append({
            "dataset": "20_newsgroups",
            "experiment": name,
            "components": n_comp,
            "f1_micro": f1_score(test_news_labels, pred, average="micro"),
            "f1_macro": f1_score(test_news_labels, pred, average="macro"),
            "f1_weighted": f1_score(test_news_labels, pred, average="weighted")
        })

df_news_pos_lsa = pd.DataFrame(news_pos_lsa_results).sort_values("f1_micro", ascending=False)
df_news_pos_lsa

,dataset,experiment,components,f1_micro,f1_macro,f1_weighted
2,20_newsgroups,N+J_tfidf,300,0.700192,0.700729,0.700800
1,20_newsgroups,N+J+V_tfidf,100,0.698911,0.699712,0.699845
3,20_newsgroups,N+J+V_tfidf,300,0.696348,0.696605,0.696621
0,20_newsgroups,N+J_tfidf,100,0.689942,0.690469,0.690559
